In [ ]:
import ee
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from datetime import datetime
from sklearn.model_selection import train_test_split

# Initialize Earth Engine
try:
    ee.Initialize(project='ee-chandanagalgali')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='ee-chandanagalgali')

# Define Mumbai shapefile
mumbai = ee.FeatureCollection("projects/ee-chandanagalgali/assets/mumbai_ward_shape_file")

# Fetch Sentinel-2 image
def fetch_sentinel2_image(start_date, end_date):
    collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterDate(start_date, end_date) \
        .filterBounds(mumbai) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    image = collection.median().select(["B2", "B3", "B4", "B8"]).clip(mumbai)
    return image

# Fetch NO2 image
def fetch_no2_image(start_date, end_date):
    collection = ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_NO2") \
        .select('tropospheric_NO2_column_number_density') \
        .filterDate(start_date, end_date)
    image = collection.mean().clip(mumbai)
    return image

# Sample training data
def sample_training_data(start_date, end_date, scale=100):
    sentinel2_image = fetch_sentinel2_image(start_date, end_date)
    no2_image = fetch_no2_image(start_date, end_date)

    combined_image = sentinel2_image.addBands(no2_image)
    samples = combined_image.sample(
        region=mumbai,
        scale=scale,
        numPixels=5000,
        geometries=False
    ).getInfo()

    predictors = []
    target = []

    for feature in samples['features']:
        props = feature['properties']
        if all(band in props for band in ["B2", "B3", "B4", "B8"]) and 'tropospheric_NO2_column_number_density' in props:
            predictors.append([props["B2"], props["B3"], props["B4"], props["B8"]])
            target.append(props['tropospheric_NO2_column_number_density'])

    return np.array(predictors), np.array(target)

# Build U-Net Model for Scalar Output
def build_unet_model(input_shape):
    inputs = tf.keras.layers.Input(shape=input_shape)

    # Encoder
    c1 = tf.keras.layers.Conv2D(16, (3, 3), activation='relu', padding='same')(inputs)
    c2 = tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same')(c1)

    # Bottleneck
    c3 = tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c2)

    # Decoder
    c4 = tf.keras.layers.Conv2DTranspose(32, (3, 3), activation='relu', padding='same')(c3)
    c5 = tf.keras.layers.Conv2DTranspose(16, (3, 3), activation='relu', padding='same')(c4)

    # Predict single scalar output
    outputs = tf.keras.layers.GlobalAveragePooling2D()(c5)
    outputs = tf.keras.layers.Dense(1, activation='linear')(outputs)

    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Train the model
def train_model(predictors, target):
    scaler = StandardScaler()
    predictors = scaler.fit_transform(predictors).reshape(-1, 2, 2, 1)  # Normalize and reshape
    target = target.reshape(-1, 1)  # Reshape as 1D for scalar predictions

    x_train, x_val, y_train, y_val = train_test_split(predictors, target, test_size=0.2, random_state=42)

    model = build_unet_model((2, 2, 1))  # Input shape for U-Net
    model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=20, batch_size=32)
    return model, scaler

if __name__ == "__main__":
    start_date = '2018-06-28'
    end_date = datetime.now().strftime('%Y-%m-%d')

    print("Sampling training data...")
    predictors, target = sample_training_data(start_date, end_date, scale=100)

    print("Training the U-Net model...")
    model, scaler = train_model(predictors, target)

    print("Saving model and scaler...")
    model.save('/content/NO2_unet_model.h5')
    np.save('/content/predictor_scaler_mean.npy', scaler.mean_)
    np.save('/content/predictor_scaler_scale.npy', scaler.scale_)

Sampling training data...
Training the U-Net model...
Epoch 1/20
123/123 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 7.2009e-06 - mae: 0.0013 - val_loss: 9.5327e-09 - val_mae: 7.1130e-05
Epoch 2/20
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 7.8350e-09 - mae: 6.6958e-05 - val_loss: 5.7626e-09 - val_mae: 5.6743e-05
Epoch 3/20
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 4.7264e-09 - mae: 5.1777e-05 - val_loss: 3.9797e-09 - val_mae: 4.6993e-05
Epoch 4/20
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 3.4257e-09 - mae: 4.4521e-05 - val_loss: 3.3654e-09 - val_mae: 4.3865e-05
Epoch 5/20
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 2.7293e-09 - mae: 4.0150e-05 - val_loss: 2.5877e-09 - val_mae: 3.8573e-05
Epoch 6/20
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 2.3218e-09 - mae: 3.7255e-05 - val_loss: 2.9313e-09 - val_mae: 4.0795e-05
Epoch 7/20
123/123 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 2.7457e-09 - mae: 3.9409e-05 - val_loss: 2.4022e-09 - val_mae: 3.7034e-05
Epoch 8/20


Saving model and scaler...


In [ ]:
import ee
import tensorflow as tf
import numpy as np

# Initialize Earth Engine
try:
    ee.Initialize(project='ee-chandanagalgali')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='ee-chandanagalgali')

# Define or import custom metrics/losses
@tf.keras.utils.register_keras_serializable()
def custom_mse(y_true, y_pred):
    return tf.reduce_mean(tf.square(y_true - y_pred))

# Load trained model and scalers
model = tf.keras.models.load_model(
    '/content/NO2_unet_model.h5',
    custom_objects={
        'custom_mse': custom_mse,
        'mse': tf.keras.losses.MeanSquaredError()
    }
)
scaler_mean = np.load('/content/predictor_scaler_mean.npy')
scaler_scale = np.load('/content/predictor_scaler_scale.npy')

# Use the exact Mumbai shapefile geometry
def get_mumbai_geometry():
    print("Using the exact Mumbai shapefile outline.")
    return ee.FeatureCollection("projects/ee-chandanagalgali/assets/mumbai_ward_shape_file").geometry()

mumbai_geometry = get_mumbai_geometry()

# Fetch image with fallbacks
def fetch_image(user_date):
    start_date = ee.Date(user_date)
    end_date = start_date.advance(1, 'day')

    # Sentinel-2
    s2_image = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterBounds(mumbai_geometry) \
        .filterDate(start_date, end_date) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30)) \
        .select(['B2', 'B3', 'B4', 'B8'])
    if s2_image.size().getInfo() > 0:
        print("Using Sentinel-2 data.")
        return s2_image.median()

    # MODIS Fallback
    print("No Sentinel-2 data. Using MODIS data.")
    modis_image = ee.ImageCollection("MODIS/061/MCD43A4") \
        .filterBounds(mumbai_geometry) \
        .filterDate(start_date, end_date) \
        .select(['Nadir_Reflectance_Band1', 'Nadir_Reflectance_Band4', 'Nadir_Reflectance_Band3', 'Nadir_Reflectance_Band2']) \
        .median().rename(['B2', 'B3', 'B4', 'B8'])
    return modis_image

# Divide region into smaller tiles
def generate_tiles(geometry, num_tiles=4):
    print("Dividing region into smaller tiles...")
    bounds = geometry.bounds().coordinates().get(0).getInfo()
    lon_min, lat_min = bounds[0]
    lon_max, lat_max = bounds[2]

    lon_step = (lon_max - lon_min) / num_tiles
    lat_step = (lat_max - lat_min) / num_tiles

    tiles = []
    for i in range(num_tiles):
        for j in range(num_tiles):
            tile = ee.Geometry.Rectangle([
                lon_min + i * lon_step,
                lat_min + j * lat_step,
                lon_min + (i + 1) * lon_step,
                lat_min + (j + 1) * lat_step
            ])
            tiles.append(tile)
    return tiles

# Process a tile and make predictions
def process_tile(tile, image):
    try:
        sampled_data = image.sample(
            region=tile,
            scale=100,  # Resolution for sampling
            geometries=False,
            numPixels=1000
        ).getInfo()

        inputs = []
        for feature in sampled_data['features']:
            props = feature['properties']
            inputs.append([props['B2'], props['B3'], props['B4'], props['B8']])

        inputs = np.array(inputs)
        if inputs.size == 0:
            return None, None

        # Normalize inputs
        inputs = (inputs - scaler_mean) / scaler_scale

        # Reshape for U-Net model
        inputs = inputs.reshape(-1, 2, 2, 1)  # Match U-Net input shape
        predictions = model.predict(inputs).flatten()

        # Take mean prediction for this tile
        mean_prediction = np.clip(predictions.mean(), 0, 0.0005)
        return mean_prediction, tile
    except Exception as e:
        print(f"Error processing tile: {e}")
        return None, None

# Predict and export NO2 layer
def predict_no2(user_date):
    try:
        print(f"Processing NO2 for {user_date}.")
        image = fetch_image(user_date)
        if image is None:
            raise ValueError("No valid image data available.")

        print(f"Bands in the image: {image.bandNames().getInfo()}")
        tiles = generate_tiles(mumbai_geometry, num_tiles=4)
        predictions = []

        for idx, tile in enumerate(tiles):
            print(f"Processing tile {idx + 1} of {len(tiles)}...")
            mean_prediction, processed_tile = process_tile(tile, image)
            if mean_prediction is not None:
                predictions.append(ee.Image.constant(mean_prediction).clip(processed_tile).rename('Downscaled_NO2'))

        if not predictions:
            print("No valid predictions generated.")
            return

        # Combine tiles into final image
        print("Combining predictions into final image...")
        downscaled_no2 = ee.ImageCollection(predictions).mosaic().clip(mumbai_geometry)

        # Export Downscaled NO2 Layer
        print("Exporting Downscaled_NO2 layer...")
        task = ee.batch.Export.image.toAsset(
            image=downscaled_no2,
            description='Downscaled_NO2',
            assetId='projects/ee-chandanagalgali/assets/downscaled_NO2_final',
            region=mumbai_geometry,
            scale=100,  # Export at 100m resolution
            maxPixels=1e13
        )
        task.start()
        print("Export task started. Monitor progress in GEE.")

    except Exception as e:
        print(f"Error processing NO2 pixel-wise: {e}")
        print("Failed to create downscaled NO₂ map.")

if __name__ == "__main__":
    user_date = input("Enter the date for prediction (YYYY-MM-DD): ")
    predict_no2(user_date)
    print("Processing complete.")

Using the exact Mumbai shapefile outline.
Enter the date for prediction (YYYY-MM-DD): 2023-12-10
Processing NO2 for 2023-12-10.
No Sentinel-2 data. Using MODIS data.
Bands in the image: ['B2', 'B3', 'B4', 'B8']
Dividing region into smaller tiles...
Processing tile 1 of 16...
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
Processing tile 2 of 16...
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
Processing tile 3 of 16...
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Processing tile 4 of 16...
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
Processing tile 5 of 16...
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
Processing tile 6 of 16...
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
Processing tile 7 of 16...
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
Processing tile 8 of 16...
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
Processing tile 9 of 16...
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Processing tile 10 of 16...
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Processing tile 11 of 16...
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Processing tile 12 of 